# Cuaderno de evaluación de GlucyLas cuatro figuras con las que se juzga el modelo, y el porqué de cada una.**Este cuaderno no guarda sus salidas a propósito.** Una figura guardada en el archivo se quedacon los números de la corrida en que se guardó, y el día que cambie el clasificador o el modeloseguirá enseñando lo de antes sin avisar. Ya pasó una vez con la rejilla de Clarke. Para tenerlas figuras en archivo, la orden del final las deja en `artifacts/figuras/`.**Lo que decide algo no está aquí.** Está en `src/eval/cuaderno.py` y `src/eval/dibujos.py`, quepasan por el estilo y tienen pruebas. Un cuaderno se rompe sin que nadie se entere hasta quealguien lo abre, probablemente la semana de la entrega.

## De dónde salen los datosPrefiere OhioT1DM, en `data/`, y cae al generador de juguete si no está. **Si cae, lo dice encada figura**: una corrida sintética no es un resultado, y una figura sin esa advertencia acabapegada en un documento como si lo fuera (riesgo R-8).Mientras no haya modelo entrenado, lo que se evalúa es la **persistencia**: «dentro de mediahora será igual que ahora». No es un relleno. Es la línea base que el protocolo pone como piso—un modelo que no le gana no justifica su existencia—, así que es exactamente la figura contrala que se comparará M-1. Tener el listón puesto antes de medir es el orden correcto.

In [ ]:
from src.eval import dibujos
from src.eval.cuaderno import escribir_figuras, evaluacion_de_referencia, serie_de_evaluacion

serie, origen = serie_de_evaluacion("data")
evaluacion = evaluacion_de_referencia(serie, origen)

print(f"origen: {origen} · {len(serie)} lecturas del sensor")
veredicto = "sí" if evaluacion.es_reportable else f"NO — {evaluacion.subtitulo}"
print(f"¿va al documento final?: {veredicto}")
evaluacion.resultado

## 1. La rejilla de error de ClarkeEl criterio principal del proyecto: **A+B ≥ 95 %**. Clasifica el error por su consecuenciaclínica y no por su tamaño, porque un error de 30 mg/dL cerca de 70 puede mandar a alguien a unahipoglucemia y el mismo error cerca de 200 no cambia ninguna decisión.El fondo se pinta **evaluando el clasificador sobre una malla**, no trazando rectas a mano. Esla única forma de que el dibujo no pueda discrepar del número que lo acompaña: `src/eval/clarke.py`cambió dos veces en este proyecto, y la última vez movió 19 900 pares de B a C. Unas rectasescritas aparte habrían seguido diciendo lo de antes.

In [ ]:
dibujos.rejilla_de_clarke(
    evaluacion.reales,
    evaluacion.predichos,
    titulo=f"Rejilla de Clarke a 30 min · {evaluacion.subtitulo}",
)

## 2. Lo medido contra lo predichoLo que un número no enseña: **si el modelo llega tarde**. Un RMSE de 15 mg/dL puede ser unacurva que sigue bien la forma o una que la sigue con diez minutos de retraso, y la segunda nosirve para anticipar nada.Línea continua para lo medido y punteada para lo predicho, igual que en la app y por el mismomotivo: una persona medio dormida tiene que distinguirlas sin leer la leyenda.

In [ ]:
hasta = min(len(evaluacion.minutos), 72)  # seis horas: más no se lee en una figura
dibujos.medido_contra_predicho(
    evaluacion.minutos[:hasta],
    evaluacion.reales[:hasta],
    evaluacion.predichos[:hasta],
    titulo=f"Lo medido y lo predicho a 30 min · {evaluacion.subtitulo}",
)

## 3. Contra las líneas base del protocoloCinco obligatorias: persistencia, extrapolación lineal, Ridge, Random Forest y LSTM. Lo quequede a la derecha de la línea de la persistencia está perdiendo contra «dentro de media horaserá igual que ahora».Hoy la única barra es la persistencia, porque no hay modelo entrenado. El día que lo haya, bastaagregar su RMSE al diccionario.

In [ ]:
dibujos.contra_las_lineas_base(
    evaluacion.rmse_por_modelo,
    el_del_proyecto=min(evaluacion.rmse_por_modelo, key=evaluacion.rmse_por_modelo.get),
)

## 4. El error por origen del datoLa figura del riesgo **R-2**. El modelo aprende de doce personas con sensor, y no hay razón parasuponer que generaliza a alguien con diabetes tipo 2 insulinodependiente en México. Lo quereduce el riesgo en parte es poder ver si el sistema funciona **peor precisamente para quien notiene sensor**, que es el usuario con menos recursos. Un promedio único lo esconde.Mientras no haya predicciones sobre lecturas manuales y por foto, la figura se arma con el únicoorigen que hay: sirve para ver la forma que tendrá, no para concluir nada.

In [ ]:
dibujos.error_por_origen(
    {"sensor": evaluacion.resultado.mae},
    n_por_origen={"sensor": evaluacion.resultado.n},
)

## Las cuatro en archivo, para el documentoLo mismo sin abrir Jupyter: `python -m src.eval.cuaderno --salida artifacts/figuras`.

In [ ]:
escribir_figuras(evaluacion, "artifacts/figuras")